In [17]:
def display_board(board):
    """Prints the board in the required +-------+ format."""
    print("+-------+-------+-------+")
    for row in board:
        print("|       |       |       |")
        print(f"|   {row[0]}   |   {row[1]}   |   {row[2]}   |")
        print("|       |       |       |")
        print("+-------+-------+-------+")

In [18]:
def make_list_of_free_fields(board):
    """Returns a list of tuples (row, col) for all empty squares."""
    free_fields = []
    for r in range(3):
        for c in range(3):
            # If the square contains a digit (1-9), it is free
            if board[r][c] not in ['X', 'O']:
                free_fields.append((r, c))
    return free_fields

In [19]:
def victory_for(board, sign):
    """Checks if the player with 'sign' has 3 in a row."""
    # Check Rows
    for r in range(3):
        if board[r][0] == board[r][1] == board[r][2] == sign:
            return True
    # Check Columns
    for c in range(3):
        if board[0][c] == board[1][c] == board[2][c] == sign:
            return True
    # Check Diagonals
    if board[0][0] == board[1][1] == board[2][2] == sign:
        return True
    if board[0][2] == board[1][1] == board[2][0] == sign:
        return True
    
    return False

In [20]:
from random import randrange
def enter_move(board):
    """Asks the user for their move, validates it, and updates the board."""
    while True:
        try:
            move = int(input("Enter your move (1-9): "))
            if move < 1 or move > 9:
                print("Error: Field number must be between 1 and 9.")
                continue
            
            # Convert 1-9 to row/col indices
            row = (move - 1) // 3
            col = (move - 1) % 3
            
            # Check if the square is occupied by 'X' or 'O'
            if board[row][col] in ['X', 'O']:
                print("Error: That field is already occupied! Try again.")
                continue
                
            board[row][col] = 'O' # Human is always 'O'
            break 
        except ValueError:
            print("Error: Please enter a valid integer.")

In [21]:
def draw_move_random(board):
    """The computer makes a random move using randrange."""
    free_fields = make_list_of_free_fields(board)
    
    if free_fields:
        # Pick a random index from the list of available (row, col) tuples
        random_index = randrange(len(free_fields))
        row, col = free_fields[random_index]
        
        board[row][col] = 'X' # Computer is always 'X'
        print(f"Computer (Random) chose square: {(row * 3) + col + 1}")

In [22]:
def minimax(board, depth, is_maximizing):
    """The recursive engine that calculates the best possible move."""
    # Base cases: Does someone win in this imaginary scenario?
    if victory_for(board, 'X'):
        return 1
    if victory_for(board, 'O'):
        return -1
    if not make_list_of_free_fields(board):
        return 0

    if is_maximizing:
        best_score = -float('inf')
        for row, col in make_list_of_free_fields(board):
            temp_val = board[row][col] # Save current value (the number)
            board[row][col] = 'X'      # Try the move
            score = minimax(board, depth + 1, False)
            board[row][col] = temp_val # Undo the move (Backtrack)
            best_score = max(score, best_score)
        return best_score
    else:
        best_score = float('inf')
        for row, col in make_list_of_free_fields(board):
            temp_val = board[row][col]
            board[row][col] = 'O'
            score = minimax(board, depth + 1, True)
            board[row][col] = temp_val
            best_score = min(score, best_score)
        return best_score

In [23]:
def draw_move_intelligent(board):
    """The computer uses Minimax to find the optimal move."""
    best_score = -float('inf')
    best_move = None
    
    print("Computer (AI) is calculating the best move...")
    
    for row, col in make_list_of_free_fields(board):
        temp_val = board[row][col]
        board[row][col] = 'X'
        score = minimax(board, 0, False)
        board[row][col] = temp_val # Undo
        
        if score > best_score:
            best_score = score
            best_move = (row, col)
            
    if best_move:
        r, c = best_move
        board[r][c] = 'X'

In [24]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. INITIALIZE GLOBAL STATE ---
board = [[str(3 * j + i + 1) for i in range(3)] for j in range(3)]
current_turn = 'O' # Human starts as O
output = widgets.Output()

# Create buttons 1-9
buttons = [widgets.Button(description=str(i+1), 
                          layout=widgets.Layout(width='60px', height='60px')) 
           for i in range(9)]

# Mode Selector UI
mode_selector = widgets.Dropdown(
    options=[('Human vs Random', 'random'), 
             ('Human vs AI (Minimax)', 'ai'), 
             ('Human vs Human', 'human')],
    value='ai',
    description='Game Mode:',
)

# --- 2. CORE FUNCTIONS ---

def reset_game(change=None):
    """Resets the board and visuals when mode changes or game ends."""
    global board, current_turn
    board = [[str(3 * j + i + 1) for i in range(3)] for j in range(3)]
    current_turn = 'O'
    
    with output:
        clear_output()
        if mode_selector.value in ['random', 'ai']:
            board[1][1] = 'X' # AI starts in middle
            print("Computer started in the middle (Square 5).")
        else:
            print("Human vs Human: Player O starts.")
            
    for b in buttons:
        b.disabled = False
    sync_visuals()

def sync_visuals():
    """Updates colors and text for all buttons."""
    for i in range(9):
        r, c = i // 3, i % 3
        buttons[i].description = board[r][c]
        if board[r][c] == 'X':
            buttons[i].style.button_color = 'salmon'
        elif board[r][c] == 'O':
            buttons[i].style.button_color = 'lightblue'
        else:
            buttons[i].style.button_color = None

def check_end_game(sign):
    """Checks for win or tie and disables buttons if game over."""
    if victory_for(board, sign):
        with output: print(f"🎉 Player {sign} Wins!")
        for b in buttons: b.disabled = True
        return True
    if not make_list_of_free_fields(board):
        with output: print("🤝 It's a Tie!")
        for b in buttons: b.disabled = True
        return True
    return False

# --- 3. CLICK HANDLER ---

def on_button_clicked(b):
    global current_turn
    with output:
        idx = int(b.description) - 1 if b.description.isdigit() else -1
        if idx == -1: return # Square occupied
        
        row, col = idx // 3, idx % 3
        
        # --- MODE 1 & 2: VS COMPUTER ---
        if mode_selector.value in ['random', 'ai']:
            # Player O Move
            board[row][col] = 'O'
            sync_visuals()
            if check_end_game('O'): return
            
            # Computer Move
            if mode_selector.value == 'random':
                draw_move_random(board)
            else:
                draw_move_intelligent(board)
            sync_visuals()
            check_end_game('X')

        # --- MODE 3: HUMAN VS HUMAN ---
        else:
            board[row][col] = current_turn
            sync_visuals()
            if check_end_game(current_turn): return
            # Switch turn
            current_turn = 'X' if current_turn == 'O' else 'O'
            print(f"Next turn: Player {current_turn}")

# --- 4. ASSEMBLE AND DISPLAY ---

for b in buttons:
    b.on_click(on_button_clicked)

# Link dropdown to reset function
mode_selector.observe(reset_game, names='value')

# Layout
grid = widgets.GridBox(buttons, layout=widgets.Layout(grid_template_columns="repeat(3, 70px)"))
ui = widgets.VBox([mode_selector, grid, output])

# Initialize visual state
reset_game()
display(ui)